# Tutorial 2: Arithmetic Secret Sharing
Arithmetic secret sharing is used in secure two party computation, where each participant receives a shared value of data. Through sharing, it ensures that the data will not leak information during the computation process. Currently, our model and functionality are designed based on semi honest participants.
To use arithmetic secret sharing for secure two party computation, we need to import the following package

In [1]:
# import the libraries
from crypto.mpc.party import SemiHonestCS
from crypto.primitives.arithmetic_secret_sharing.arithmetic_secret_sharing import ArithmeticSecretSharing
from crypto.tensor.RingTensor import RingTensor
from crypto.primitives.beaver.beaver import BeaverOfflineProvider

import torch

```SemiHonestCS "is the semi honest two participant we use," Arithmetic SecretSharing "is the package we primarily use for secure two party computation," RingSensor "is the main data structure we use, and" BeaverOfflineProvider "is the triplet provider we use for multiplication and other stages of arithmetic secret sharing. We use" BeaverOfflineProvider "to simulate trusted third-party auxiliary operation data.

## Party
We first set the participants to participate in the computation, considering secure two party computation, we need two participants - Server and Client.
In the process of setting up participants, we need to set the address and port of each participant. In order to perform subsequent operations, we also need to set the data type, accuracy, and beaver triplet provider for the operation. If you want to perform size comparison operations, don't forget to set the comparison auxiliary parameter provider.
Please note: If you want to perform a series of operations on floating-point numbers, be sure to set dtype to float and scale to 65536. If it is an integer operation, it needs to be set to int and 1 respectively, otherwise an error will occur!
Here, we demonstrate using multithreading, where the server and client are run in two separate files in practical applications. Please refer to/ Debug/crypto/primitives/ass/ass_server. py and/ debug/crypto/primitives/ass/ass_client.py

In [2]:
import threading

# set Server
server = SemiHonestCS(type='server')
server.set_address('127.0.0.1')  # set server address
server.set_port(8989)  # set server port
# set data type
server.set_dtype('float')
server.set_scale(65536)
# set beaver provider
server.set_beaver_provider(BeaverOfflineProvider())
# set compare key provider
server.set_compare_key_provider()
# When use OfflineProvider, do not forget to load beaver triples
server.beaver_provider.load_triples(server, 2)


def set_server():
    # CS connect
    server.connect()


# set Client
client = SemiHonestCS(type='client')
client.set_address('127.0.0.1')  # set server address
client.set_port(8989)  # set server port
# set data type
client.set_dtype('float')
client.set_scale(65536)
# set beaver provider
client.set_beaver_provider(BeaverOfflineProvider())
# set compare key provider
client.set_compare_key_provider()
# When use OfflineProvider, do not forget to load beaver triples
client.beaver_provider.load_triples(client, 2)


def set_client():
    # CS connect
    client.connect()

server_thread = threading.Thread(target=set_server)
client_thread = threading.Thread(target=set_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()

TCPServer waiting for connection ......
successfully connected to server: 127.0.0.1
TCPServer successfully connected by :('127.0.0.1', 1414)


## Secret Sharing


In [3]:
# data belong to server
x = RingTensor.convert_to_ring(torch.tensor([[1.0, 2.0], [3.0, 4.0]]))
# data belong to client
y = RingTensor.convert_to_ring(torch.tensor([[-1.0, 2.0], [4.0, 3.0]]))

# split x into 2 parts
X = ArithmeticSecretSharing.share(x, 2)

# split y into 2 parts
Y = ArithmeticSecretSharing.share(y, 2)

# server shares x1 to client
server.send_ring_tensor(X[1])
# client receives x1 from server
x1 = client.receive_ring_tensor()

# client shares y0 to server
client.send_ring_tensor(Y[0])
# server receives y0 from client
y0 = server.receive_ring_tensor()

# convert RingTensor to ASS
# server
shared_x_0 = ArithmeticSecretSharing(X[0], server)
shared_y_0 = ArithmeticSecretSharing(y0, server)

print("\n shared x in server:", shared_x_0)
print("\n shared y in server:", shared_y_0)

# client
shared_x_1 = ArithmeticSecretSharing(x1, client)
shared_y_1 = ArithmeticSecretSharing(Y[1], client)
print("\n shared x in client:", shared_x_1)
print("\n shared y in client:", shared_y_1)


 shared x in server: [ArithmeticSecretSharing
 value:tensor([[ 3342201674875939594, -1942694563752708514],
        [-2431887468552533812, -6570006491377421398]], device='cuda:0'),
 party:0]

 shared y in server: [ArithmeticSecretSharing
 value:tensor([[ 8426368768686845735, -2503340824033103912],
        [-9028967629627733203, -1670224480252442706]], device='cuda:0'),
 party:0]

 shared x in client: [ArithmeticSecretSharing
 value:tensor([[-3342201674875874058,  1942694563752839586],
        [ 2431887468552730420,  6570006491377683542]], device='cuda:0'),
 party:1]

 shared y in client: [ArithmeticSecretSharing
 value:tensor([[-8426368768686911271,  2503340824033234984],
        [ 9028967629627995347,  1670224480252639314]], device='cuda:0'),
 party:1]


## Secret Restoring


In [4]:
# restore share_x
# server
def restore_server():
    restored_x = shared_x_0.restore()
    real_x = restored_x.convert_to_real_field()
    print("\n x after restoring:", real_x)

# client
def restore_client():
    shared_x_1.restore()

server_thread = threading.Thread(target=restore_server)
client_thread = threading.Thread(target=restore_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


 x after restoring: tensor([[1., 2.],
        [3., 4.]], device='cuda:0')


## Operations


#### Arithmetic Operations

In [5]:
# Addition
# restore result
def addition_server():
    res_0 = shared_x_0 + shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\nAddition", result_restored)

def addition_client():
    res_1 = shared_x_1 + shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=addition_server)
client_thread = threading.Thread(target=addition_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


Addition tensor([[0., 4.],
        [7., 7.]], device='cuda:0')


In [6]:
# Subtraction
# restore result
def subtraction_server():
    res_0 = shared_x_0 + shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\nSubtraction", result_restored)

def subtraction_client():
    res_1 = shared_x_1 + shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=subtraction_server)
client_thread = threading.Thread(target=subtraction_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


Subtraction tensor([[0., 4.],
        [7., 7.]], device='cuda:0')


In [7]:
# Multiplication
# restore result
def multiplication_server():
    res_0 = shared_x_0 * shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\nMultiplication", result_restored)

def multiplication_client():
    res_1 = shared_x_1 * shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=multiplication_server)
client_thread = threading.Thread(target=multiplication_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


Multiplication tensor([[-1.,  4.],
        [12., 12.]], device='cuda:0')


注意：由于所有用到的beaver三元组都是在离线阶段生成的，所以在运算矩阵乘法前不要忘了生成所需的矩阵beaver triples

In [8]:
# Matrix Multiplication
# generate matrix beaver triples before operation
BeaverOfflineProvider().gen_matrix_beaver(x.shape, y.shape)
# restore result
def matrix_multiplication_server():
    res_0 = shared_x_0 @ shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\nMatrix Multiplication", result_restored)

def matrix_multiplication_client():
    res_1 = shared_x_1 @ shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=matrix_multiplication_server)
client_thread = threading.Thread(target=matrix_multiplication_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


Matrix Multiplication tensor([[ 7.,  8.],
        [13., 18.]], device='cuda:0')


#### Comparison Operations


In [9]:
# Less than
def less_than_server():
    res_0 = shared_x_0 < shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\n(x < y)", result_restored)

def less_than_client():
    res_1 = shared_x_1 < shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=less_than_server)
client_thread = threading.Thread(target=less_than_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


(x < y) tensor([[0., 0.],
        [1., 0.]], device='cuda:0')


In [10]:
# Less than or equal
def less_equal_server():
    res_0 = shared_x_0 <= shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\n(x <= y)", result_restored)

def less_equal_client():
    res_1 = shared_x_1 <= shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=less_equal_server)
client_thread = threading.Thread(target=less_equal_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


(x <= y) tensor([[0., 1.],
        [1., 0.]], device='cuda:0')


In [11]:
# Greater than
def greater_than_server():
    res_0 = shared_x_0 > shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\n(x > y)", result_restored)

def greater_than_client():
    res_1 = shared_x_1 > shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=greater_than_server)
client_thread = threading.Thread(target=greater_than_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


(x > y) tensor([[1., 0.],
        [0., 1.]], device='cuda:0')


In [12]:
# Greater than or equal
def greater_equal_server():
    res_0 = shared_x_0 >= shared_y_0
    result_restored = res_0.restore().convert_to_real_field()
    print("\n(x >= y)", result_restored)

def greater_equal_client():
    res_1 = shared_x_1 >= shared_y_1
    res_1.restore()

server_thread = threading.Thread(target=greater_equal_server)
client_thread = threading.Thread(target=greater_equal_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()


(x >= y) tensor([[1., 1.],
        [0., 1.]], device='cuda:0')
